In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.4 Complex Vectors, Hermitian, Unitary, and Normal Matrices

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Volume III — Eigenvalues and Spectral Theory",
    number="3.4",
    title="Complex Vectors, Hermitian, Unitary, and Normal Matrices",
    blurb="One conjugate transpose replaces every transpose, symmetry becomes "
    "Hermitian, orthogonal becomes unitary — and the exact condition for an "
    "orthonormal eigenbasis turns out to be neither of those, but normality. "
    "Then a qubit, which is a unit vector in C-two.",
    difficulty="advanced",
    estimate="105–135 min",
)

## Notebook overview

Everything in this volume so far has been real, and the one place that hurt was
[§3.1](eigenvalues-diagonalization.ipynb): a rotation has no real eigenvector,
so its eigenvalues came back as $\pm i$ and the theory had to look away. Moving
to $\mathbb{C}^n$ fixes that, and the cost is a single change — every
$A^{\top}$ becomes $A^{*} = \bar{A}^{\top}$, the **conjugate** transpose.

With that one substitution, the whole of [§3.2](spectral-theorem.ipynb)
transfers. Symmetric becomes **Hermitian** ($A = A^{*}$) with real eigenvalues
and an orthonormal eigenbasis; orthogonal becomes **unitary** ($U^{*}U = I$)
with $|\lambda| = 1$ and exact norm preservation. `np.linalg.eigh` already
handles both.

The interesting part is what the exact condition turns out to be. Hermitian is
sufficient for a unitary diagonalization but not necessary — a unitary matrix
has one too, and its eigenvalues are not real. The precise condition is
**normality**, $AA^{*} = A^{*}A$, and it is strictly weaker than either. The
clean way to see this is the **Schur decomposition** $A = QTQ^{*}$, which
exists for *every* square matrix with $T$ upper triangular, and whose $T$ is
diagonal exactly when $A$ is normal. Measured on a four-matrix suite the
separation is total: the commutator vanishes identically for the three normal
matrices and equals 1 for the fourth, and the Schur $T$ follows suit.

The notebook ends in physics, where all of this is load-bearing. A qubit is a
unit vector in $\mathbb{C}^2$, a quantum gate is a $2\times2$ unitary, and
observables are Hermitian *because* the spectral theorem makes their
eigenvalues real — which is what lets a measurement return a number. The Pauli
matrices satisfy an algebra we verify to zero error, and the Bloch sphere turns
the whole of $\mathbb{C}^2$ into a picture.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own. Complex spectra make the
> ordering problem worse than usual, so several checks here are deliberately
> built to need no sorting at all.

> **Scope.** Horn and Johnson {cite}`horn2013` Chapter 2 for Schur and
> normality; Trefethen and Bau {cite}`trefethen1997` Lecture 24; Axler
> {cite}`axler2024` Chapter 7. For the quantum side, Nielsen and Chuang is the
> standard reference; nothing here needs more than $\mathbb{C}^2$.

## Theory in brief

### The complex inner product

On $\mathbb{C}^n$ the inner product carries a conjugate:

```{math}
:label: eq-herm-inner
\langle \mathbf{x}, \mathbf{y}\rangle = \bar{\mathbf{x}}^{\top}\mathbf{y}
  = \sum_i \bar{x}_i y_i ,
\qquad
\|\mathbf{x}\|^2 = \langle\mathbf{x},\mathbf{x}\rangle = \sum_i |x_i|^2 .
```

The conjugate is not decoration. Without it,
$\mathbf{x}^{\top}\mathbf{x}$ for $\mathbf{x} = (1, i)$ is $1 + i^2 = 0$: a
nonzero vector of zero length, and no geometry at all.
[§0.3](../00-machine/vectors-norms-inner-products.ipynb) measured exactly this.
The price is that {eq}`eq-herm-inner` is **conjugate-symmetric** rather than
symmetric, $\langle\mathbf{y},\mathbf{x}\rangle =
\overline{\langle\mathbf{x},\mathbf{y}\rangle}$, and conjugate-linear in its
first argument.

### Hermitian and unitary

Write $A^{*} = \bar{A}^{\top}$. Then $A$ is **Hermitian** if

```{math}
:label: eq-herm-hermitian
A = A^{*} ,
```

and **unitary** if

```{math}
:label: eq-herm-unitary
U^{*}U = UU^{*} = I .
```

A Hermitian matrix has real eigenvalues and an orthonormal eigenbasis — the
proof of [§3.2](spectral-theorem.ipynb) goes through verbatim with $\top$
replaced by $*$. A unitary matrix preserves the inner product, hence every
length and angle, so all its eigenvalues satisfy $|\lambda| = 1$: they lie on
the unit circle rather than on the real line.

### Schur, and the exact condition

Every square complex matrix — with no hypothesis whatsoever — has a **Schur
decomposition**

```{math}
:label: eq-herm-schur
A = QTQ^{*},
\qquad Q \text{ unitary},\; T \text{ upper triangular},
```

with the eigenvalues of $A$ on the diagonal of $T$. This is the computable
substitute for the Jordan form, and it is what LAPACK actually produces.

The decomposition is *diagonal* exactly when

```{math}
:label: eq-herm-normal
AA^{*} = A^{*}A ,
```

which is **normality**. Hence the sharpest form of the spectral theorem:

```{math}
:label: eq-herm-spectral-normal
A = Q\Lambda Q^{*} \text{ for some unitary } Q
\quad\Longleftrightarrow\quad
A \text{ is normal.}
```

Hermitian matrices are normal (both products are $A^2$); unitary matrices are
normal (both are $I$); real symmetric matrices are the real special case. But
normality is strictly weaker than either, and a matrix can be normal with
eigenvalues neither real nor on the unit circle — a circulant, for instance.

### Circulants and the DFT

A **circulant** is constant along each diagonal wrapped cyclically, so it is
determined by its first column $\mathbf{c}$. Every circulant is normal, and
more than that, every circulant is diagonalized by the *same* unitary matrix:
with $F_{jk} = e^{-2\pi ijk/n}/\sqrt{n}$ the unitary DFT of
[§2.5](../02-orthogonality/function-space-bases.ipynb),

```{math}
:label: eq-herm-circulant
FCF^{*} = \operatorname{diag}\bigl(\hat{\mathbf{c}}\bigr),
\qquad \hat{\mathbf{c}} = \texttt{np.fft.fft}(\mathbf{c}) .
```

The eigenvectors do not depend on $\mathbf{c}$ at all, which is why convolution
becomes multiplication and why the FFT is the workhorse it is.

### The Pauli matrices

The three matrices

$$
\sigma_x = \begin{bmatrix}0&1\\1&0\end{bmatrix},\quad
\sigma_y = \begin{bmatrix}0&-i\\i&0\end{bmatrix},\quad
\sigma_z = \begin{bmatrix}1&0\\0&-1\end{bmatrix}
$$

are Hermitian *and* unitary, traceless, and each has eigenvalues $\pm1$. With
$I$ they span all $2\times2$ Hermitian matrices, and they close under
multiplication:

```{math}
:label: eq-herm-pauli
\sigma_i\sigma_j = \delta_{ij}I + i\sum_k \varepsilon_{ijk}\sigma_k ,
```

with $\varepsilon$ the Levi-Civita symbol. Every entry involved is $0$, $\pm1$
or $\pm i$, so this identity holds in floating point *exactly*.

### Qubits and the Bloch sphere

A **qubit** is a unit vector in $\mathbb{C}^2$,

```{math}
:label: eq-herm-qubit
|\psi\rangle = \alpha|0\rangle + \beta|1\rangle,
\qquad |\alpha|^2 + |\beta|^2 = 1 ,
```

and a **gate** is a $2\times2$ unitary, unitary precisely so that the state
stays normalised. Two complex numbers are four real parameters; normalisation
removes one and an unobservable global phase removes another, leaving two — the
surface of a sphere. The coordinates are the expectation values of the Paulis,

```{math}
:label: eq-herm-bloch
\mathbf{r} = \bigl(\langle\sigma_x\rangle, \langle\sigma_y\rangle,
                   \langle\sigma_z\rangle\bigr),
\qquad \langle\sigma\rangle = \langle\psi|\sigma|\psi\rangle ,
```

and $\|\mathbf{r}\| = 1$ for every pure state. A gate is then a **rotation** of
that sphere, which is the whole reason the picture is worth having.

---
## Setup

Data and instruments only: the worked matrices, a careful spectrum sorter,
and a Bloch-coordinates reader. Nothing here is any exercise's lesson.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import schur

from ecp import animate, validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps
np.set_printoptions(precision=5, suppress=True, linewidth=120)

# The four-matrix suite. Three are normal for three different reasons; the
# fourth is not normal at all, and every test below is applied to all four.
H_MAT = np.array([[2.0, 1 - 2j, 0.0],
                  [1 + 2j, 3.0, 1j],
                  [0.0, -1j, 5.0]])                      # Hermitian
_TH = np.pi / 5
U_MAT = np.array([[np.cos(_TH), -np.sin(_TH), 0.0],
                  [np.sin(_TH), np.cos(_TH), 0.0],
                  [0.0, 0.0, 1.0]], dtype=complex)       # unitary (a rotation)
C_MAT = np.array([[1, 2, 3],
                  [3, 1, 2],
                  [2, 3, 1]], dtype=complex)             # circulant: normal
N_MAT = np.array([[1, 1, 0],
                  [0, 1, 1],
                  [0, 0, 1]], dtype=complex)             # NOT normal
SUITE = {"Hermitian": H_MAT, "unitary": U_MAT,
         "circulant": C_MAT, "non-normal": N_MAT}

# The Pauli matrices and the gates built from them.
I2 = np.eye(2, dtype=complex)
SIGMA_X = np.array([[0, 1], [1, 0]], dtype=complex)
SIGMA_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
SIGMA_Z = np.array([[1, 0], [0, -1]], dtype=complex)
PAULIS = [SIGMA_X, SIGMA_Y, SIGMA_Z]


# instrument: a sorter, not a method — complex spectra order ambiguously
# (this notebook's own lesson), and this helper pins ONE ordering so
# comparisons elsewhere are reproducible.
def spectrum_sorted(A):
    """Eigenvalues in a canonical order: by real part, ties broken by imaginary.

    np.sort_complex sorts on the real part alone and leaves ties in input order,
    so a conjugate pair can come back either way round and a naive comparison of
    two spectra fails on identical answers. Sorting lexicographically fixes that
    — but where a check can avoid sorting altogether, it does.
    """
    w = np.linalg.eigvals(A)
    return np.array(sorted(w, key=lambda z: (round(z.real, 10), round(z.imag, 10))))


# instrument: a coordinates reader for the two-level figure — extraction,
# not construction.
def bloch_vector(psi):
    """The Bloch coordinates of Eq. 10: the three Pauli expectation values."""
    return np.array([float(np.real(psi.conj() @ s @ psi)) for s in PAULIS])

## Exercise 1: The conjugate, and what goes wrong without it

{eq}`eq-herm-inner` differs from the real inner product by one conjugation, and
the whole of complex linear algebra depends on it. Without the conjugate,
"length" stops being a length: the form $\mathbf{x}^{\top}\mathbf{x}$ is still
bilinear and symmetric, but it is no longer positive, and vectors of zero
length that are not zero appear immediately.

**Part a)** For $\mathbf{x} = (1, i)$ and $\mathbf{y} = (2 - i, 3i)$, compute
both forms: the unconjugated $\mathbf{x}^{\top}\mathbf{x}$ with `x @ x`, and
the conjugated $\bar{\mathbf{x}}^{\top}\mathbf{x}$ with `np.vdot(x, x)`.
Report both. The first is exactly $0$ — a nonzero vector of zero length — and
the second is $2$.

**Part b)** Confirm `np.vdot(x, y)` conjugates its **first** argument, which is
the physics convention and the opposite of what a naive reading of `np.dot`
would suggest. Check `np.vdot(x, y) == np.conj(np.vdot(y, x))` exactly, which
is the conjugate symmetry of {eq}`eq-herm-inner`.

**Part c)** Confirm positivity properly. Draw $10^{4}$ random complex vectors
as `rng.standard_normal((10_000, 4)) + 1j * rng.standard_normal((10_000, 4))`
and confirm that $\langle\mathbf{x},\mathbf{x}\rangle$ is real (imaginary part
below $10^{-15}$) and strictly positive for every one. Then confirm the
unconjugated form is complex for at least $99\%$ of them.

**Part d)** Confirm $\|\mathbf{x}\|^2 = \sum|x_i|^2$ agrees with
`np.linalg.norm(x)**2` to $10^{-13}$ over the same $10^{4}$ vectors, so the
`numpy` norm is the conjugated one.

**Part e)** Confirm conjugate-linearity in the first slot and linearity in the
second: for the same $\mathbf{x}, \mathbf{y}$ and $c = 2 + 3i$, check
$\langle c\mathbf{x}, \mathbf{y}\rangle = \bar{c}\langle\mathbf{x},\mathbf{y}\rangle$
and $\langle \mathbf{x}, c\mathbf{y}\rangle = c\langle\mathbf{x},\mathbf{y}\rangle$,
both to $10^{-15}$. The asymmetry is unavoidable: it is what buys positivity.

In [ ]:
# (solution hidden on the public site)


### Validation 1

The failing form is computed and reported rather than described, because
"$\mathbf{x}^{\top}\mathbf{x}$ can be zero for $\mathbf{x} \ne \mathbf{0}$" is
the entire justification for the conjugate and it is worth seeing as a number.

In [ ]:
validate.check(
    unconj == 0.0 and abs(conj - 2.0) < 1e-15,
    "for x = (1, i) the unconjugated form is 0 and the conjugated one is 2 (Eq. 1)",
    f"x^T x = {unconj}, conj(x)^T x = {conj}: without the conjugate a nonzero "
    "vector has zero length and there is no geometry",
)
validate.check(
    imag_max < 1e-15 and all_pos,
    "<x,x> is real and strictly positive for all 10,000 random vectors (Eq. 1)",
    f"largest |imaginary part| {imag_max:.2e}, smallest value "
    f"{inner_self.real.min():.4f}",
)
validate.check(
    frac_complex > 0.99,
    "while the unconjugated form is complex for essentially all of them",
    f"{100*frac_complex:.2f}% have a nonzero imaginary part, so it cannot be a "
    "squared length",
)
validate.check(
    norm_gap < 1e-13 and sym_gap < 1e-15,
    "np.linalg.norm uses the conjugated form, and it is conjugate-symmetric",
    f"||x||^2 agrees to {norm_gap:.2e}; <x,y> = conj(<y,x>) to {sym_gap:.1e}",
)
validate.check(
    lin1 < 1e-15 and lin2 < 1e-15,
    "and it is conjugate-linear in the first slot, linear in the second (Eq. 1)",
    f"errors {lin1:.1e} and {lin2:.1e}: the asymmetry is what buys positivity",
)

## Exercise 2: Hermitian and unitary: the spectral theorem, one symbol at a time

Replace every $\top$ by $*$ in [§3.2](spectral-theorem.ipynb) and the proofs go
through unchanged. This exercise confirms that on two concrete matrices, and
picks out the one place the two cases differ: where the eigenvalues live.

The Hermitian matrix is

$$
H = \begin{bmatrix} 2 & 1-2i & 0\\ 1+2i & 3 & i\\ 0 & -i & 5\end{bmatrix},
$$

available as `H_MAT`, and the unitary one is the $3\times3$ rotation by
$\pi/5$ in the first two coordinates, `U_MAT`.

**Part a)** Confirm $H = H^{*}$ **exactly** with
`np.abs(H_MAT - H_MAT.conj().T).max()`, which is $0$, and note the structure:
a Hermitian matrix has a real diagonal and conjugate-paired off-diagonal
entries.

**Part b)** Take `lam_H, Q_H = np.linalg.eigh(H_MAT)` — `eigh` handles the
complex Hermitian case too — and confirm the eigenvalues are real, that
`np.linalg.eigvals` (the general routine) gives $|\operatorname{Im}\lambda| <
10^{-15}$, and that $Q^{*}Q = I$ to $10^{-14}$. The eigenvalues are
$0.1263, 4.1497, 5.7240$.

**Part c)** Confirm the reconstruction $H = Q\Lambda Q^{*}$ to $10^{-14}$,
using `Q_H @ np.diag(lam_H) @ Q_H.conj().T`. Note that $\Lambda$ is real while
$Q$ is complex — a Hermitian matrix has real eigenvalues and genuinely complex
eigenvectors.

**Part d)** Confirm $U^{*}U = I$ exactly for the rotation, and that it
preserves the norm: for $10^{3}$ random complex vectors, check
$\|U\mathbf{x}\| = \|\mathbf{x}\|$ to $10^{-14}$. This is the complex form of
the property that made $QR$ stable in
[§2.2](../02-orthogonality/gram-schmidt-qr.ipynb).

**Part e)** Confirm $|\lambda| = 1$ for every eigenvalue of $U$, to
$10^{-14}$, and report them. They are $e^{\pm i\pi/5}$ and $1$ — genuinely
complex, on the unit circle rather than the real line. This is the one place
the Hermitian and unitary cases part company: both give an orthonormal
eigenbasis, but the spectra live on perpendicular curves in $\mathbb{C}$.

**Part f)** Confirm the physical reading. An observable in quantum mechanics is
required to be Hermitian, and the reason is Part b): a measurement returns a
real number, so the eigenvalues must be real. Check that
$\langle\psi|H|\psi\rangle$ is real to $10^{-15}$ for $10^{3}$ random unit
vectors in $\mathbb{C}^3$ — the expectation value of a Hermitian operator is
real for *every* state, not only for eigenstates.

In [ ]:
# (solution hidden on the public site)


### Validation 2

The expectation-value check is the one that carries the physics: it confirms
$\langle\psi|H|\psi\rangle$ is real for *every* state and not merely for
eigenstates, which is the actual reason observables are required to be
Hermitian. It is also the Rayleigh quotient of
[§3.2](spectral-theorem.ipynb) wearing a different notation, so the bracket
result transfers too.

In [ ]:
validate.check(
    herm_gap == 0.0 and unit_gap < 1e-15,
    "H = H* and U* U = I, both exactly (Eqs. 2, 3)",
    f"|H - H*| = {herm_gap:.1e}, |U*U - I| = {unit_gap:.1e}",
)
validate.check(
    imag_H < 1e-15,
    "a Hermitian matrix has real eigenvalues, from the general routine (Eq. 2)",
    f"largest |Im lambda| = {imag_H:.2e} from np.linalg.eigvals, which assumes "
    "nothing; eigh returns them real by construction",
)
validate.close(
    Q_H.conj().T @ Q_H, np.eye(3),
    "with an orthonormal eigenbasis: Q* Q = I (Eq. 6)",
    rtol=0.0, atol=1e-14,
)
validate.close(
    Q_H @ np.diag(lam_H) @ Q_H.conj().T, H_MAT,
    "and H = Q Lambda Q* with Lambda real and Q complex (Eq. 6)",
    rtol=0.0, atol=100 * EPS * float(np.linalg.norm(H_MAT, 2)),
)
validate.check(
    norm_pres < 1e-14 and mod_gap < 1e-14,
    "a unitary preserves every norm, so all its eigenvalues have |lambda| = 1",
    f"||Ux|| = ||x|| to {norm_pres:.2e} over 1000 complex vectors; "
    f"||lambda| - 1| <= {mod_gap:.2e}. Hermitian spectra lie on the real line, "
    "unitary spectra on the unit circle",
)
validate.check(
    expect_imag < 1e-15 and in_range,
    "and <psi|H|psi> is real for EVERY state, which is why observables are Hermitian",
    f"largest |imaginary part| {expect_imag:.2e} over 1000 random unit states, "
    f"all landing inside [{lam_H[0]:.4f}, {lam_H[-1]:.4f}] — the Rayleigh "
    "bracket of 3.2, transferred",
)

## Exercise 3: Normality is the exact condition, and Schur shows it

Hermitian gives a unitary diagonalization; so does unitary; neither is
necessary. {eq}`eq-herm-spectral-normal` names the exact condition, and
{eq}`eq-herm-schur` is how to see it: every square matrix has a Schur form
$A = QTQ^{*}$, and $T$ is diagonal precisely when $A$ is normal.

The suite is the four matrices in `SUITE`: $H$ (Hermitian), $U$ (unitary), the
circulant $C = \left[\begin{smallmatrix}1&2&3\\3&1&2\\2&3&1\end{smallmatrix}\right]$
— normal but neither Hermitian nor unitary — and
$N = \left[\begin{smallmatrix}1&1&0\\0&1&1\\0&0&1\end{smallmatrix}\right]$,
which is not normal.

**Part a)** For each matrix compute the commutator norm
$\|AA^{*} - A^{*}A\|_{\max}$. Report all four. The first three are at rounding
level and the fourth is exactly 1.

Two of the three are *exactly* zero, and it is worth being precise about which.
$H$ and $C$ have Gaussian-integer entries, so every product and sum in
$AA^{*}$ stays a Gaussian integer well below $2^{53}$ and no rounding occurs at
all. The rotation's entries are $\cos(\pi/5)$ and $\sin(\pi/5)$, which are not
exactly representable — so even $\cos^2 + \sin^2 = 1$ is a floating-point
question, and its commutator lands on $0$ or on $10^{-17}$ *depending on the
BLAS*: this machine and the CI runner disagree about it.

Confirm the claims separately, and gate only what is provable: that all three
are at rounding level, and that the two Gaussian-integer ones are exactly zero
on any IEEE-754 machine. The rotation's value is reported, never gated — an
exactness that one library delivers and another does not is a fact about the
library.

**Part b)** Confirm that the three normal ones are normal for three *different*
reasons: $H$ is Hermitian but not unitary, $U$ is unitary but not Hermitian,
and $C$ is neither. Check each of those six claims. Normality is genuinely
weaker than either condition.

**Part c)** Compute the Schur form of each with
`scipy.linalg.schur(A, output="complex")`, which returns `(T, Z)` with
$A = ZTZ^{*}$. Confirm the reconstruction to $10^{-14}$ for all four —
{eq}`eq-herm-schur` needs no hypothesis, so it must work even on $N$.

**Part d)** Report $\|T - \operatorname{diag}(T)\|_{\max}$ for each. It is
below $2\times10^{-15}$ for the three normal matrices and exactly $1$ for $N$.
Confirm the separation is at least 10 orders of magnitude, which is
{eq}`eq-herm-spectral-normal` measured.

**Part e)** Confirm $Z$ is unitary in all four cases to $10^{-14}$. Schur
always gives an *orthonormal* basis; what it cannot always give is a diagonal
$T$. That is the precise sense in which the non-normal case is harder, and
[§3.5](schur-jordan-nonnormality.ipynb) is about what the off-diagonal part
does.

**Part f)** Confirm the corollary that makes normality worth naming: for a
normal matrix the Schur $T$ is diagonal, so its diagonal *is* the spectrum and
$Z$ is an orthonormal eigenbasis. Verify for $C$ that
$\|CZ - Z\Lambda\| < 10^{-14}$ with $\Lambda = \operatorname{diag}(T)$, and
that $Z$'s columns are eigenvectors of a matrix that is neither Hermitian nor
unitary.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

The separation between the normal and non-normal Schur forms is the whole
content of {eq}`eq-herm-spectral-normal`, and it is checked as a *ratio* so
that the claim is "these differ by fifteen orders" rather than "one of them is
small". The Schur reconstruction is checked on all four, including the
non-normal one, because that decomposition is the one with no hypothesis.

In [ ]:
validate.check(
    all(comm[k] < 1e-15 for k in ["Hermitian", "unitary", "circulant"])
    and comm["non-normal"] > 0.5,
    "AA* = A*A for the three normal matrices and not the fourth (Eq. 5)",
    f"commutator norms {[f'{comm[k]:.1e}' for k in SUITE]} against "
    f"{comm['non-normal']:.1f} for the non-normal one",
)
validate.check(
    comm["Hermitian"] == 0.0 and comm["circulant"] == 0.0,
    "and for the two with Gaussian-integer entries it is EXACTLY zero",
    f"Hermitian {comm['Hermitian']:.0f} and circulant {comm['circulant']:.0f}: "
    "products and sums of Gaussian integers below 2^53 are exact on any IEEE-754 "
    f"machine. The rotation's is {comm['unitary']:.2e} here — whether that lands "
    "on 0 or on 1e-17 is a property of the BLAS, so it is reported and not gated",
)
validate.check(
    max(recon_s.values()) < 1e-14,
    "the Schur form A = Z T Z* reconstructs all four, hypothesis-free (Eq. 4)",
    f"worst reconstruction error {max(recon_s.values()):.2e}, including the "
    "non-normal matrix — Schur needs nothing at all",
)
validate.check(
    max(unit_Z.values()) < 1e-14,
    "with Z unitary in every case",
    f"worst |Z*Z - I| = {max(unit_Z.values()):.2e}: Schur always gives an "
    "orthonormal basis; what it cannot always give is a diagonal T",
)
validate.check(
    separation > 1e10,
    "and T is diagonal exactly for the normal ones (Eq. 6)",
    f"off-diagonal {normal_max:.2e} for the three normal matrices against "
    f"{schur_off['non-normal']:.1e} for the fourth, a separation of "
    f"{separation:.1e}x",
)
validate.check(
    eig_res_C < 1e-13,
    "so the circulant has an orthonormal eigenbasis while being neither H nor U",
    f"||C Z - Z Lambda|| = {eig_res_C:.2e}. Normality is strictly weaker than "
    "Hermitian and strictly weaker than unitary, and it is the exact condition",
)

## Exercise 4: Every circulant is diagonalized by the same matrix

The circulant of Exercise 3 was normal, so it has *an* orthonormal eigenbasis.
{eq}`eq-herm-circulant` says something much stronger: the basis is the same for
**every** circulant of that size, and it is the DFT matrix
[§2.5](../02-orthogonality/function-space-bases.ipynb) already built. The
eigenvalues are then just the Fourier transform of the first column.

This is the structural fact behind the whole of signal processing: a circulant
matrix is a cyclic convolution, and in the Fourier basis convolution becomes
multiplication.

**Part a)** Build the unitary DFT matrix for $n = 3$ as
`F = np.exp(-2j * np.pi * np.outer(j, j) / n) / np.sqrt(n)` with
`j = np.arange(n)`, matching `np.fft`'s sign convention, and confirm
$F^{*}F = I$ to $10^{-14}$.

**Part b)** Confirm {eq}`eq-herm-circulant` directly: compute
`F @ C_MAT @ F.conj().T` and confirm it is diagonal to $10^{-14}$. Report the
diagonal.

**Part c)** Confirm the diagonal equals `np.fft.fft(C_MAT[:, 0])` — the
transform of the first **column** — to $10^{-14}$, *entry by entry and in
order*. This check needs no sorting at all, which is the point: comparing two
complex spectra by sorting them is fragile. `np.sort_complex` does sort
lexicographically, but $-1.5 \pm 0.866i$ is a **near-tie in the real part**,
and rounding breaks that tie differently in the two arrays — so the comparison
may succeed or fail depending on the last bit of $\sigma$. Report what it
gives here, and note that the CI runner gets a different answer on identical
spectra. A check whose verdict depends on the BLAS is worse than one that
always fails, because it cannot be found by running it once.

**Part d)** Confirm the eigenvector claim is $\mathbf{c}$-independent. Build
three *different* random circulants of size 6 from
`rng.standard_normal(6)` using `scipy.linalg.circulant`, and confirm the same
$F$ (now $6\times6$) diagonalizes all three to $10^{-13}$. The eigenvalues
differ; the eigenvectors do not.

**Part e)** Confirm the convolution property. For a random circulant $C$ built
from $\mathbf{c}$ and a random vector $\mathbf{v}$ of length 6, confirm
$C\mathbf{v}$ equals the cyclic convolution computed by
`np.fft.ifft(np.fft.fft(c) * np.fft.fft(v)).real` to $10^{-13}$. Multiplying by
a circulant *is* convolution, and the FFT does it in $O(n\log n)$ instead of
$O(n^2)$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

The order-free comparison is the substance of this validation. Two complex
spectra that agree as *sets* can differ arbitrarily as *arrays*, and the
demonstration that `np.sort_complex` produces a discrepancy of 1.7 on identical
data is included precisely so the reader does not later write that check.

In [ ]:
validate.close(
    F3.conj().T @ F3, np.eye(3),
    "the DFT matrix F is unitary (Eq. 7)",
    rtol=0.0, atol=1e-14,
)
validate.check(
    diag_gap < 1e-14 and ordered_gap < 1e-14,
    "F C F* is diagonal, with diagonal exactly fft(first column) (Eq. 7)",
    f"off-diagonal {diag_gap:.2e}; the diagonal matches fft entry by entry and "
    f"in order to {ordered_gap:.2e}, so no sorting convention can break it",
)
validate.check(
    True,
    "whereas the sorted comparison is fragile in a machine-dependent way",
    f"np.sort_complex gives {sorted_gap:.4f} here on identical spectra, and a "
    "different value on the CI runner. It sorts lexicographically, but rounding "
    "in the REAL parts breaks the near-tied conjugate pair differently in the "
    "two arrays, so whether the comparison succeeds depends on the last bit of "
    "sigma. That is worse than always failing, and it is why this is reported "
    "and not gated — and why the check above avoids sorting entirely",
    strict=False,
)
validate.check(
    max(od for od, _ in circ_gaps) < 1e-13
    and max(fg for _, fg in circ_gaps) < 1e-13,
    "and the SAME F diagonalizes three different 6x6 circulants (Eq. 7)",
    f"worst off-diagonal {max(od for od, _ in circ_gaps):.2e}, worst eigenvalue "
    f"gap {max(fg for _, fg in circ_gaps):.2e}: the eigenvectors do not depend "
    "on the matrix at all",
)
validate.check(
    conv_gap < 1e-13,
    "which is why multiplying by a circulant is cyclic convolution",
    f"C v matches the FFT convolution to {conv_gap:.2e}, in O(n log n) rather "
    "than O(n^2)",
)

## Exercise 5: The Pauli matrices, and an algebra that is exact in float64

The three Pauli matrices are the smallest interesting example of everything
above: each is Hermitian *and* unitary, hence normal twice over, with
eigenvalues $\pm1$ that are simultaneously real (Hermitian) and on the unit
circle (unitary). They also close under multiplication, {eq}`eq-herm-pauli`.

What makes this exercise unusual is that every entry involved is $0$, $\pm1$ or
$\pm i$, so the products are computed by multiplying and adding exactly
representable numbers. The identity therefore holds to **exactly zero**, not to
machine precision, and that is a rare enough thing to check for deliberately.

**Part a)** Confirm each of $\sigma_x, \sigma_y, \sigma_z$ is Hermitian to
exactly $0$, unitary to exactly $0$ ($\sigma^2 = I$), traceless to exactly $0$,
and has eigenvalues $\{-1, +1\}$ to $10^{-15}$.

**Part b)** Verify {eq}`eq-herm-pauli` for all nine pairs $(i, j)$. Build the
Levi-Civita symbol $\varepsilon_{ijk}$ explicitly as a $3\times3\times3$ array,
form the right-hand side $\delta_{ij}I + i\sum_k\varepsilon_{ijk}\sigma_k$, and
report the largest discrepancy over all nine. It is exactly $0$.

**Part c)** Read off the two consequences. The $i = j$ cases give
$\sigma_i^2 = I$; the $i \ne j$ cases give the anticommutation
$\sigma_i\sigma_j = -\sigma_j\sigma_i$. Confirm the anticommutator
$\{\sigma_i, \sigma_j\} = 2\delta_{ij}I$ for all nine pairs, exactly.

**Part d)** Confirm the Paulis and $I$ span the $2\times2$ Hermitian matrices.
For $10^{3}$ random Hermitian $2\times2$ matrices, built as $M = (B + B^{*})/2$
from complex Gaussian $B$, recover the coefficients as
$a_\mu = \tfrac12\operatorname{tr}(\sigma_\mu M)$ with
$\sigma_0 = I$, and confirm $M = \sum_\mu a_\mu\sigma_\mu$ to $10^{-14}$ with
every $a_\mu$ real to $10^{-15}$.

**Part e)** Confirm the gate algebra. Build the Hadamard
$H_d = \tfrac{1}{\sqrt2}\left[\begin{smallmatrix}1&1\\1&-1\end{smallmatrix}\right]$,
the phase gate $S = \operatorname{diag}(1, i)$, and CNOT as the $4\times4$
permutation swapping the last two basis states. Verify $H_d^2 = I$,
$\sigma_x^2 = I$, $S^2 = \sigma_z$, $\mathrm{CNOT}^2 = I$ and
$H_d\sigma_zH_d = \sigma_x$, all to $10^{-15}$, and confirm all six matrices
are unitary. The last identity is the statement that the Hadamard exchanges the
$z$ and $x$ axes of the Bloch sphere, which Exercise 6 draws.

In [ ]:
# (solution hidden on the public site)


### Validation 5

`atol=0` appears here, which it almost never should. It is justified because
every quantity involved is exactly representable: the Pauli entries are $0$,
$\pm1$ and $\pm i$, and products of those involve no rounding at all. Where the
Hadamard's $1/\sqrt2$ enters, the tolerance goes back to $10^{-15}$.

In [ ]:
validate.check(
    all(h == 0.0 and u == 0.0 and tr == 0.0 for h, u, tr, _ in pauli_props),
    "each Pauli matrix is Hermitian, unitary and traceless, EXACTLY (Eqs. 2, 3)",
    "all three properties hold to exactly zero — no rounding occurs, because "
    "every entry is 0, +/-1 or +/-i",
)
validate.check(
    max(e for _, _, _, e in pauli_props) < 1e-15,
    "with eigenvalues -1 and +1: real (Hermitian) and on the unit circle (unitary)",
    f"largest deviation {max(e for _, _, _, e in pauli_props):.1e}; the Paulis "
    "are normal twice over",
)
validate.check(
    pauli_worst == 0.0 and anticomm_worst == 0.0,
    "the algebra sigma_i sigma_j = delta_ij I + i eps_ijk sigma_k is exact (Eq. 8)",
    f"largest discrepancy over all nine pairs is {pauli_worst:.0f}, and the "
    f"anticommutator identity holds to {anticomm_worst:.0f}",
)
validate.check(
    span_gap < 1e-14 and coeff_imag < 1e-15,
    "and (I, sigma_x, sigma_y, sigma_z) spans the 2x2 Hermitian matrices",
    f"1000 random Hermitian matrices reconstructed to {span_gap:.2e} with all "
    f"coefficients real to {coeff_imag:.2e}",
)
validate.check(
    max(gate_ids.values()) < 1e-15 and gates_unitary < 1e-15,
    "the gate identities hold, including H Z H = X (Eq. 3)",
    f"worst identity {max(gate_ids.values()):.1e}, worst unitarity "
    f"{gates_unitary:.1e}. H Z H = X says the Hadamard exchanges the z and x "
    "axes of the Bloch sphere",
)

## Exercise 6: A qubit is a unit vector in $\mathbb{C}^2$

{eq}`eq-herm-qubit` and {eq}`eq-herm-bloch` turn the abstract picture into a
concrete one. A pure state is a unit vector in $\mathbb{C}^2$; its Bloch vector
is the triple of Pauli expectation values; and that triple always lies on the
unit sphere. A gate, being unitary, moves the state around the sphere without
ever leaving it — which is what "unitary preserves the norm" looks like when
drawn.

The state is
$|\psi_0\rangle = \cos(0.6)|0\rangle + e^{0.9i}\sin(0.6)|1\rangle$.

**Part a)** Build $|\psi_0\rangle$ as
`np.array([np.cos(0.6), np.exp(1j * 0.9) * np.sin(0.6)])` and confirm
$\||\psi_0\rangle\| = 1$ to $10^{-15}$.

**Part b)** Compute its Bloch vector with `bloch_vector`, defined above as the
three expectation values $\langle\psi|\sigma_i|\psi\rangle$. Confirm each is
real to $10^{-15}$ — they must be, since the Paulis are Hermitian and Exercise
2 showed a Hermitian expectation is real — and report
$\mathbf{r} = (0.5794, 0.7301, 0.3624)$.

**Part c)** Confirm $\|\mathbf{r}\| = 1$ to $10^{-14}$. Every pure state sits
on the sphere's *surface*; the interior corresponds to mixed states, which need
a density matrix rather than a vector.

**Part d)** Apply the rotation gate
$R_z(t) = \cos(t/2)I - i\sin(t/2)\sigma_z$ for 120 values of
$t \in [0, 2\pi]$. Confirm each $R_z(t)$ is unitary to $10^{-15}$, that
$\|\mathbf{r}(t)\| = 1$ throughout to $10^{-14}$, and — the geometric content
— that $r_z$ is **constant** to $10^{-15}$ while $(r_x, r_y)$ traces a circle
of constant radius $0.9320$ to $10^{-15}$. A rotation about $z$ is exactly
that.

**Part e)** Confirm the period. $R_z(2\pi) = -I$, not $I$: a $2\pi$ rotation
returns the state to $-|\psi\rangle$, which is physically the same state
because global phase is unobservable. Check $R_z(2\pi) = -I$ to $10^{-15}$ and
that the Bloch vector at $t = 2\pi$ equals the one at $t = 0$ to $10^{-14}$.
The Bloch sphere quotients out exactly the phase that makes spin-$\tfrac12$
need $4\pi$.

**Part f)** Animate the Bloch vector precessing under $R_z(t)$ over one full
period, drawn on a wireframe sphere.

```{admonition} With your assistant
:class: tip
Every $2\times2$ unitary is a rotation of the Bloch sphere, and the
correspondence can be made explicit: for each unitary $U$ there is a $3\times3$
real rotation $R$ with $R_{ij} = \tfrac12\operatorname{tr}(\sigma_iU\sigma_jU^{*})$,
and this is the two-to-one map $SU(2) \to SO(3)$. Ask your assistant to write
`bloch_rotation(U)` returning that $R$. Then check it against the mathematics
rather than against a picture: verify (i) $R^{\top}R = I$ and $\det R = +1$ to
$10^{-14}$ for the Hadamard, for $S$, and for $R_z(0.7)$; (ii) that $R$ acting
on a Bloch vector agrees with applying $U$ to the state and recomputing, to
$10^{-14}$, over 100 random states; and (iii) the two-to-one part — that $U$
and $-U$ give the *same* $R$, to exactly zero, which is why a $2\pi$ rotation
flips the state's sign but not its Bloch vector. The check is yours.
```

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

The animation's validation checks the **data** — the 120 Bloch vectors — and
never the animation object, following the course's rule. The two geometric
invariants (constant height, constant radius) are what make "a rotation about
$z$" a checkable statement rather than a description of a picture.

In [ ]:
validate.close(
    np.array([norm0]), np.array([1.0]),
    "the qubit state is a unit vector in C^2 (Eq. 9)",
    rtol=0.0, atol=1e-15,
)
validate.check(
    expect_imag_p < 1e-15 and abs(np.linalg.norm(r0) - 1.0) < 1e-14,
    "its Bloch vector is real and lies on the unit sphere (Eq. 10)",
    f"expectations imaginary to {expect_imag_p:.1e}; ||r|| = "
    f"{np.linalg.norm(r0):.15f}. Pure states are on the surface; the interior "
    "needs a density matrix",
)
validate.check(
    max(unit_gaps) < 1e-15 and sphere_gap < 1e-14,
    "every R_z(t) is unitary, so the state never leaves the sphere (Eq. 3)",
    f"worst |R*R - I| = {max(unit_gaps):.1e}, worst |||r|| - 1| = "
    f"{sphere_gap:.1e} over 120 gates",
)
validate.check(
    rz_drift < 1e-15 and rad_drift < 1e-15,
    "and it really is a rotation about z: constant height, constant radius",
    f"r_z drifts by {rz_drift:.1e} from {r0[2]:.6f}, and the (r_x, r_y) radius "
    f"by {rad_drift:.1e} from {rad_xy:.6f}",
)
validate.check(
    minus_I < 1e-15 and period_gap < 1e-14,
    "R_z(2 pi) = -I, yet the Bloch vector returns exactly (Eqs. 3, 10)",
    f"|R_z(2 pi) + I| = {minus_I:.1e} and |r(2 pi) - r(0)| = {period_gap:.1e}: "
    "the sphere quotients out the global phase that makes spin-1/2 need 4 pi",
)

---
## Notebook summary

**One conjugate, and the geometry survives.** For $\mathbf{x} = (1, i)$ the
unconjugated $\mathbf{x}^{\top}\mathbf{x}$ is **exactly 0** — a nonzero vector
of zero length — while $\bar{\mathbf{x}}^{\top}\mathbf{x} = 2$. Over $10^{4}$
random complex vectors the conjugated form was real to $10^{-16}$ and strictly
positive every time, while the unconjugated one was complex for $100\%$ of
them.

**[§3.2](spectral-theorem.ipynb) transfers with $\top \to *$.** The Hermitian
$H$ satisfied $H = H^{*}$ exactly, `eigvals` returned
$|\operatorname{Im}\lambda| < 4\times10^{-16}$, $Q^{*}Q = I$ to $10^{-15}$ and
$Q\Lambda Q^{*} = H$ to $10^{-15}$ — with $\Lambda$ real and $Q$ genuinely
complex. The unitary satisfied $U^{*}U = I$ exactly, preserved the norm to
$9\times10^{-16}$ over $10^{3}$ complex vectors, and had all $|\lambda| = 1$ to
$3\times10^{-16}$. And $\langle\psi|H|\psi\rangle$ was real to $10^{-16}$ for
every one of $10^{3}$ random states — the reason observables must be Hermitian,
and the Rayleigh bracket of [§3.2](spectral-theorem.ipynb) in new notation.

**Normality is the exact condition.** Across the four-matrix suite the
commutator $AA^{*} - A^{*}A$ was at rounding level for the Hermitian, the
unitary and the circulant, and exactly 1 for the fourth — and **exactly zero**
for the two with Gaussian-integer entries, where no rounding can occur. For the
rotation, whose $\cos(\pi/5)$ makes even $\cos^2+\sin^2=1$ a rounding question,
this machine and the CI runner disagree — so that one is reported, not gated. The Schur form
reconstructed all four to $10^{-15}$ — it needs no hypothesis — with $Z$
unitary throughout, and its off-diagonal part was below $2\times10^{-15}$ for
the three normal matrices against $1$ for the non-normal one, a separation of
$7\times10^{14}$. The circulant is neither Hermitian nor unitary and still has
an orthonormal eigenbasis, which is what "strictly weaker" means.

**Every circulant shares one eigenbasis.** $FCF^{*}$ was diagonal to
$2\times10^{-15}$ with diagonal equal to `np.fft.fft` of the first column
**entry by entry and in order**, so no sorting convention enters. The same $F$
diagonalized three different $6\times6$ circulants, and $C\mathbf{v}$ matched
the FFT convolution to $10^{-15}$.

**A comparison that fails unpredictably.** The same spectra compared through
`np.sort_complex` differ on identical data — by $1.7$ on this machine and by
nothing at all on the CI runner. It sorts lexicographically, but
$-1.5 \pm 0.866i$ is a near-tie in the real part and rounding breaks it
differently in the two arrays. A check whose verdict depends on the BLAS cannot
be found by running it once, which is why the notebook uses the order-free form
and reports this one rather than gating it.

**An algebra that is exact.** Each Pauli matrix is Hermitian, unitary and
traceless to exactly zero, with eigenvalues $\pm1$; the identity
$\sigma_i\sigma_j = \delta_{ij}I + i\varepsilon_{ijk}\sigma_k$ held over all
nine pairs to **exactly zero**, as did the anticommutator. With $I$ they
reconstructed $10^{3}$ random Hermitian $2\times2$ matrices to $10^{-15}$ with
real coefficients. The gate identities $H^2 = I$, $S^2 = Z$,
$\mathrm{CNOT}^2 = I$ and $HZH = X$ all held to $2\times10^{-16}$.

**A qubit, drawn.** The Bloch vector of the test state was
$(0.5794, 0.7301, 0.3624)$ with $\|\mathbf{r}\| = 1$ exactly. Under $R_z(t)$
across 120 steps the norm stayed 1 to $2\times10^{-16}$, $r_z$ was constant to
$2\times10^{-16}$ and the $(r_x, r_y)$ radius constant to $4\times10^{-16}$.
$R_z(2\pi) = -I$ rather than $+I$, yet the Bloch vector returned exactly:
the sphere quotients out precisely the global phase that makes spin-$\tfrac12$
need $4\pi$.

**Methods introduced.** `np.vdot` and the conjugate transpose `.conj().T`,
`np.linalg.eigh` on complex Hermitian input, `scipy.linalg.schur` with
`output="complex"`, `scipy.linalg.circulant`, the unitary DFT matrix, the
Levi-Civita symbol as an explicit array, Pauli expansion coefficients by
$\tfrac12\operatorname{tr}(\sigma_\mu M)$, and Bloch coordinates.

## Outlook

- **What the off-diagonal part does.** Schur gave $T$ with a nonzero
  off-diagonal for the non-normal matrix, and this notebook only measured that
  it was nonzero. [§3.5](schur-jordan-nonnormality.ipynb) asks what it *costs*:
  eigenvalues that no longer control the behaviour, transient growth in a matrix
  whose spectrum says decay, and pseudospectra as the honest picture.
- **Normal matrices are exactly the ones where eigenvalues tell the truth.**
  For a normal matrix $\|A\|_2 = \max|\lambda_i|$ and the condition number of
  the eigenvector matrix is 1. Both fail for non-normal matrices, and how badly
  is measured by $\operatorname{cond}(X)$ — the Bauer–Fike constant, which is
  the general replacement for the Weyl bound of
  [§3.2](spectral-theorem.ipynb).
- **Circulants are the beginning of structured linear algebra.**
  [§6.3](../06-structure/circulant-toeplitz-fft.ipynb) takes
  {eq}`eq-herm-circulant` seriously: Toeplitz matrices embed in circulants,
  so an $O(n^2)$ product becomes $O(n\log n)$, and the FFT stops being a
  transform and becomes a factorization.
- **From one qubit to many.** Two qubits live in $\mathbb{C}^2 \otimes
  \mathbb{C}^2 = \mathbb{C}^4$, which is why CNOT was $4\times4$, and $n$
  qubits in $\mathbb{C}^{2^n}$ — the exponential that makes simulation hard and
  makes low-rank tensor formats necessary.
  [§7.3](../07-tensors/tensor-trains-mps.ipynb) builds the tensor-train format
  that makes many-qubit states tractable when the entanglement is bounded.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()